### 멀티모달 모델
기존의 LLM이 텍스트라는 단일 모드만 처리했다면, 멀티모달 모델은 여러 가지 형태의 정보를 함께 처리할 수 있다.

1. CLIP (Contrastive Language-Image Pre-training)
- OpenAI에서 발표한 모델로, 텍스트와 이미지를 하나의 공간에서 연결하는 혁신적인 방법론이다.
- 기존 AI가 이미지를 "강아지", "고양이"라는 정해진 클래스로만 배웠다면, CLIP은 자연어 설명을 통해 이미지의 맥락을 배운다.

2. 공동 임베딩 공간(Shared Embedding Space)
- 멀티모달 모델은 텍스트와 이미지를 각각의 특징 벡터로 변환한 뒤, 이를 하나의 좌표 평면에 배치한다.
- 텍스트 인코더: "해변을 달리는 골든 리트리버"라는 문장을 벡터로 변환.
- 이미지 인코더: 실제 강아지 사진을 벡터로 변환.
- 대조 학습(Contrastive Learning): 관련 있는 텍스트와 이미지 벡터는 가까워지도록 유도하고, 관련 없는 데이터끼리는 멀어지도록 학습한다.

`관련 문서: https://contents.premium.naver.com/byline/commercebn/contents/260421165257027nn`

3. 벡터 유사도 (Vector Similarity)
- AI는 두 데이터가 얼마나 비슷한지 수학적으로 계산하며, 주로 주로 코사인 유사도(Cosine Similarity)를 사용한다.
- 유사도 1에 가까움: 이 텍스트는 이미지의 정확한 설명이다.
- 유사도 0에 가까움: 이 텍스트와 이미지는 아무 상관이 없다.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

In [4]:
# 로컬경로냐 url경로냐를 구분하는 메소드를 클래스로 만들거야.
import os
import base64
from langchain_core.messages import HumanMessage

class SimpleMultiModal:
    def __init__(self, llm):
        self.llm = llm

    def stream(self, input_source, prompt="이 이미지를 설명해줘"):
        # 로컬 파일인지 확인
        # 로컬 파일이 들어왔다면 url로 변환
        if os.path.exists(input_source):
            with open(input_source, 'rb') as f:
                data = base64.b64encode(f.read()).decode("utf-8")
                image_url = f'data:image/jpeg;base64,{data}'

        # url이 들어왔다면 그냥 그대로
        else:
            image_url = input_source
            
        # 텍스트 질문 + 이미지 를 한 번에 모델에게 보내는 부분

        # 이미지를 같이 보낼 때는 content가 문자열 하나가 아니라 리스트가 됨.
        #   content=[
        #       텍스트 파트,
        #       이미지 파트
        #   ]
        #   첫 번째 파트: {"type": "text", "text": prompt}
        #   => 모델에게 줄 질문 예를 들어: "고양이와 강아지 중 어디에 더 가까운가?"
        
        #   두 번째 파트: {"type": "image_url", "image_url": {"url": image_url}}
        #   =>모델에게 보여줄 이미지입니다.
        
        #   즉, 이 메시지는 사람이 모델에게 이렇게 말하는 것과 같습니다.
        
        #   이 이미지를 보고, "고양이와 강아지 중 어디에 더 가까운가?"에 답해줘.
        #   [이미지 첨부]
        
        message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}}
        ])

        return self.llm.stream([message])        

##### 중요한 포인트가

1. self.llm은 밖에서 주입한 모델 - 클래스 안에서 모델을 직접 만들지 않고:

   def __init__(self, llm):
       self.llm = llm

    외부에서 받은 llm을 저장함. 그래서 아래처럼 씀.

  llm = ChatOpenAI(
      temperature=0.1,
      model_name="gpt-5.4-nano"
  )

  smm = SimpleMultiModal(llm)

  이렇게 하면 SimpleMultiModal은 특정 모델에 고정되지 않고 나중에 모델만 바꿔도 클래스 코드는 그대로 쓸 수 있다.

  llm = ChatOpenAI(model_name="gpt-5.4-mini")
  smm = SimpleMultiModal(llm)

  2. HumanMessage 안의 content 리스트 순서도 의미가 있음

  HumanMessage(content=[
      {"type": "text", "text": prompt},
      {"type": "image_url", "image_url": {"url": image_url}}
  ])

  여기서는 텍스트를 먼저 주고 이미지를 뒤에 줬습니다.

  의미상으로는:

  이 질문에 답해줘.
  [이미지]

  입니다.

  보통 이렇게 써도 됩니다. 반대로 이미지 먼저, 텍스트 나중도 가능하지만, 질문을 명확히 먼저 넣는 게 읽기 좋습니다.

  3. image_url이라는 이름이지만 로컬 이미지도 들어갈 수 있음

  변수 이름은 image_url인데, 실제 값은 둘 중 하나입니다.

  URL인 경우:

  https://example.com/image.jpg

  로컬 파일인 경우:

  data:image/jpeg;base64,...

  즉 여기서 말하는 image_url은 꼭 인터넷 URL만 뜻하는 게 아니라, OpenAI API가 이해할 수 있는 이미지 입력 형식 전체를 뜻
  합니다.

  4. 현재 코드는 모든 로컬 파일을 jpeg로 가정함

  image_url = f'data:image/jpeg;base64,{data}'

  이건 톰.jpg면 괜찮습니다.

  하지만 PNG 파일이면 엄밀히는 이렇게 되어야 합니다.

  data:image/png;base64,...

  지금 코드는 .png를 넣어도 jpeg라고 보냅니다. 보통 되는 경우도 있지만, 정확하게 하려면 확장자에 따라 MIME 타입을 바꾸는
  게 좋습니다.

  5. stream()의 반환값은 아직 답변 문자열이 아님

  return self.llm.stream([message])

  이건 답변 텍스트가 아니라 generator/stream 객체를 반환합니다.

  그래서 바로 출력하려면 안 되고:

  stream = smm.stream(...)
  stream_response(stream)

  처럼 순회해야 합니다.

  또는 직접 쓰면:

  for chunk in smm.stream('./images/톰.jpg'):
      print(chunk.content, end="", flush=True)

  6. 이 클래스의 역할

  SimpleMultiModal은 사실 이 작업을 감싸는 도구입니다.

  이미지 경로 또는 URL 받기
  -> 로컬 파일이면 base64로 변환
  -> HumanMessage에 텍스트 + 이미지 담기
  -> LLM에 stream으로 보내기

  즉, 매번 긴 코드를 안 쓰려고 만든 래퍼 클래스

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano"
)

smm = SimpleMultiModal(llm)
# stream = smm.stream('./images/그림/마스크.jpg')
stream = smm.stream('./images/톰.jpg', prompt="고양이와 강아지 중 어디에 더 가까운가?")
stream_response(stream)

사진 속 고양이(회색 줄무늬)가 보이니까, **고양이에 더 가깝습니다.**

In [17]:
import os
import base64
from langchain_core.messages import HumanMessage, SystemMessage

# self.llm
#   실제로 호출할 모델
# self.system_prompt
#   모델의 역할이나 답변 스타일을 정하는 시스템 지시문. 없으면 None
# self.human_prompt
#   이미지와 같이 보낼 사용자 질문. 기본값은 "이 이미지를 설명해줘"
#   예를 들면 이렇게
    # pmm = PersonalMultiModal(
    #   llm,
    #   system_prompt="너는 이미지 분석 전문가야. 짧고 정확하게 답해.",
    #   human_prompt="이 이미지에 있는 객체를 분석해줘."
    # )

class PersonalMultiModal:
    def __init__(self, llm, system_prompt=None, human_prompt="이 이미지를 설명해줘"):
        self.llm = llm
        self.system_prompt = system_prompt
        self.human_prompt = human_prompt

    # 로컬 이미지를 base64 문자열로 바꾸는 메서드
    def _encode_image(self, image_path):
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")

    def stream(self, input_source):
        if os.path.exists(input_source):
            data = self._encode_image(input_source)
            image_url = f"data:image/jpeg;base64,{data}"
        else:
            image_url = input_source

        # SystemMessage가 있을 수도 있고 없을 수도 있기 때문에 비워둠. ex)"너는 이미지 분석 전문가야." 같은거
        messages = []
        
        if self.system_prompt:
            messages.append(SystemMessage(content=self.system_prompt))

        # prompt를 함수 인자로 받지 않고, 객체가 가지고 있는:
        # self.human_prompt 를 씀. 그니까 객체를 만들 때 정한 질문을 계속 재사용
        messages.append(
            HumanMessage(content=[
                {"type": "text", "text": self.human_prompt},
                {"type": "image_url", "image_url": {"url": image_url}}
            ])
        )

        return self.llm.stream(messages)

In [18]:
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano"
)

system_prompt = """
당신은 매출표를 분석하는 친절한 경영학 박사 AI입니다.
답변 이후 사용자에게 추가적인 질문은 하지 않습니다.
"""

human_prompt = """
주어진 표는 회사의 매출표 입니다. 
당기 순이익 및 향후 인사이트를 제공하세요.
5문장 이내로 답변해줘.
"""

pmm = PersonalMultiModal(llm, system_prompt, human_prompt)

stream = pmm.stream("./images/매출표.png")
stream_response(stream)

표 기준 **당기 순이익은 168,000,000원**입니다(= 영업이익 230,000,000원 − 영업외비용 15,000,000원 − 영업외비용 42,000,000원 + 법인세차감전이익 210,000,000원 − 법인세비용 42,000,000원).  
매출(1,500,000,000원) 대비 **매출총이익은 800,000,000원(약 53.3%)**으로 원가구조가 비교적 양호합니다.  
다만 **영업이익이 230,000,000원(영업이익률 약 15.3%)**으로, 판관비(320,000,000원)가 이익을 상당 부분 잠식하고 있어 비용 통제가 핵심 과제입니다.  
향후 인사이트로는 (1) 판관비 항목 중 변동비 성격을 가진 비용을 우선 절감/효율화하고, (2) 매출총이익률을 유지하기 위해 원가(매입/제조) 관리와 가격정책을 함께 점검하는 것이 유리합니다.

• 앞의 SimpleMultiModal은 이미지를 분석할 때마다 질문을 함수에 직접 넣습니다.

  smm.stream("./images/톰.jpg", prompt="이 이미지에 있는 동물을 말해줘")
  smm.stream("./images/강아지.jpg", prompt="이 이미지에 있는 동물을 말해줘")
  smm.stream("./images/새.jpg", prompt="이 이미지에 있는 동물을 말해줘")

  이미지만 바뀌고 질문은 같은데도, 매번 prompt="..."를 써야 합니다.

  반면 PersonalMultiModal은 객체를 만들 때 질문을 미리 저장합니다.

  pmm = PersonalMultiModal(
      llm,
      system_prompt="너는 동물 사진을 분류하는 전문가야.",
      human_prompt="이 이미지에 있는 동물을 말해줘"
  )

  이제 pmm 안에 이미 이 값들이 저장돼 있습니다.

  self.system_prompt = "너는 동물 사진을 분류하는 전문가야."
  self.human_prompt = "이 이미지에 있는 동물을 말해줘"

  그래서 이미지를 분석할 때는 이미지만 넘기면 됩니다.

  pmm.stream("./images/톰.jpg")
  pmm.stream("./images/강아지.jpg")
  pmm.stream("./images/새.jpg")

  즉 “매번 프롬프트를 새로 넘기지 않아도 된다”는 말은, 함수 호출할 때마다:

  prompt="이 이미지에 있는 동물을 말해줘"

  를 반복해서 쓰지 않아도 된다는 뜻입니다.

  비유하면:

  smm.stream(이미지, 질문)

  은 매번 이미지 + 질문을 같이 주는 방식이고,

  pmm = PersonalMultiModal(llm, 질문)
  pmm.stream(이미지)

  은 질문을 객체에 미리 저장해두고, 나중에는 이미지만 바꿔 넣는 방식입니다.